## Data subset and baseline configuration.

In [1]:
!wget https://raw.githubusercontent.com/allenai/bi-att-flow/master/squad/evaluate-v1.1.py -O evaluate_v1.py
!pip install rank_bm25 -q


--2026-06-11 06:06:28--  https://raw.githubusercontent.com/allenai/bi-att-flow/master/squad/evaluate-v1.1.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3419 (3.3K) [text/plain]
Saving to: ‘evaluate_v1.py’

evaluate_v1.py      100%[===================>]   3.34K  --.-KB/s    in 0s      

2026-06-11 06:06:28 (45.3 MB/s) - ‘evaluate_v1.py’ saved [3419/3419]



In [2]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

Thu Jun 11 06:06:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P0             27W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip uninstall transformers tokenizers huggingface-hub peft \
    accelerate datasets evaluate safetensors \
    sentence-transformers -y -q

!pip install \
    "numpy<2.0" \
    transformers==4.41.0 \
    tokenizers==0.19.1 \
    huggingface-hub==0.23.4 \
    datasets==2.19.0 \
    accelerate==0.30.1 \
    evaluate==0.4.6 \
    safetensors==0.4.3 \
    rank_bm25 \
    optuna -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.23.4 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.23.4 which is incompatible.


In [4]:
import torch
import transformers
import datasets
import accelerate

In [5]:
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"accelerate   : {accelerate.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
print(f"GPU          : {torch.cuda.get_device_name(0)}")

torch        : 2.2.0+cu118
transformers : 4.41.0
datasets     : 2.19.0
accelerate   : 0.30.1
CUDA         : True
GPU          : Tesla P100-PCIE-16GB


In [6]:
try:
    import peft
    print(f"peft is installed: {peft.__version__}")
except ImportError:
    print("peft is NOT installed ✅")

peft is NOT installed ✅


In [7]:
import json
import urllib.request
import os
import re
from transformers import DistilBertTokenizerFast
from evaluate_v1 import normalize_answer, exact_match_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from rank_bm25 import BM25Okapi
# add your Hugging Face token here
os.environ["HF_TOKEN"] = ""
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# Global preprocessing storage
all_encodings = []
articles = []
train_articles = []
train_encoding_strided, train_input_ids, train_attention_masks, train_start_positions, train_end_positions = [], [], [], [], []
val_articles = []
val_encoding_strided, val_input_ids, val_attention_masks, val_start_positions, val_end_positions = [], [], [], [], []
test_articles = []
test_encoding_strided, test_input_ids, test_attention_masks, test_start_positions, test_end_positions = [], [], [], [], []

### 🔷 <font color="#9CE2FF">Sherouk's Work: SQuAD Dataset Preprocessing & Distilbert Tokenization Pipeline</font>

## Pre-processing Steps Include:
1.   Load SQuAD and inspect the JSON structure.
2.   Clean only lightly, not aggressively.
3.   Tokenize question and context together with special tokens.
4.   Use truncation and a stride/window approach.
5.   Compute answer start and end offsets.
6.   Create train/validation/test splits.

**Load SQuAD and inspect the JSON structure.**


*   Download Squad V2.0 both training and evaluation data.
*   List keys and investigate first 5 items.
*   Check that every ``answer_start`` offset in the dataset actually points to the correct position in the context string. (Sanity Check)




In [8]:
URLS = {
    "train_v1": "https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v1.1.json"
}

for name, url in URLS.items():
    filename = url.split("/")[-1]
    if not os.path.exists(filename):
        print(f"Downloading {filename} ...")
        urllib.request.urlretrieve(url, filename)
    else:
        print(f"{filename} already exists, skipping download.")

with open("train-v1.1.json", "r", encoding="utf-8") as f:
    train_v1 = json.load(f)

print(f"SQuAD v1.1 - train articles: {len(train_v1['data'])}")

train-v1.1.json already exists, skipping download.
SQuAD v1.1 - train articles: 442


In [9]:
def print_keys(obj, indent=0):
    prefix = "  " * indent
    if isinstance(obj, dict):
        for key, value in obj.items():
            print(f"{prefix}- {key}")
            print_keys(value, indent + 1)
    elif isinstance(obj, list) and len(obj) > 0:
        print_keys(obj[0], indent + 1)

print("=== SQuAD v1.1 JSON Structure ===")
print_keys(train_v1)

=== SQuAD v1.1 JSON Structure ===
- data
    - title
    - paragraphs
        - context
        - qas
            - answers
                - answer_start
                - text
            - question
            - id
- version


In [10]:
count = 0
for article in train_v1["data"]:
    for para in article["paragraphs"]:
        for qa in para["qas"]:
            print(json.dumps({
                "title": article["title"],
                "context": para["context"][:150] + "...",
                "question": qa["question"],
                "answer_text": qa["answers"][0]["text"],
                "answer_start": qa["answers"][0]["answer_start"]
            }, indent=2))
            print("-" * 60)
            count += 1
            if count == 5:
                break
        if count == 5:
            break
    if count == 5:
        break

{
  "title": "University_of_Notre_Dame",
  "context": "Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front o...",
  "question": "To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?",
  "answer_text": "Saint Bernadette Soubirous",
  "answer_start": 515
}
------------------------------------------------------------
{
  "title": "University_of_Notre_Dame",
  "context": "Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front o...",
  "question": "What is in front of the Notre Dame Main Building?",
  "answer_text": "a copper statue of Christ",
  "answer_start": 188
}
------------------------------------------------------------
{
  "title": "University_of_Notre_Dame",
  "context": "Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden sta

In [11]:
total = 0
mismatches = []
for article in train_v1["data"]:
    for para in article["paragraphs"]:
        ctx = para["context"]
        for qa in para["qas"]:
            for ans in qa["answers"]:
                total += 1
                start = ans["answer_start"]
                end = start + len(ans["text"])
                extracted = ctx[start:end]
                if extracted != ans["text"]:
                    mismatches.append({
                        "id": qa["id"],
                        "expected": ans["text"],
                        "got": extracted
                    })

print(f"Checked  : {total} answer spans")
print(f"Mismatches: {len(mismatches)}")

if mismatches:
    print("\nFirst 3 mismatches:")
    for m in mismatches[:3]:
        print(f"  id={m['id']}  expected='{m['expected']}'  got='{m['got']}'")
else:
    print("All spans aligned correctly.")

Checked  : 87599 answer spans
Mismatches: 0
All spans aligned correctly.


**Clean only lightly, not aggressively.**


*    Collapse multiple spaces into one and strips leading/trailing whitespace.
*    Remove unanswerable questions, where ``is_impossible`` field distinguishes v2.0 from v1.1, it flags questions that have no answer in the context.

In [12]:
def clean_dataset(dataset):
    for article in train_v1["data"]:
      for para in article["paragraphs"]:
          para["context"] = re.sub(r" +", " ", para["context"]).strip()
          for qa in para["qas"]:
              qa["question"] = re.sub(r" +", " ", qa["question"]).strip()

    return dataset

In [13]:
train_v1 = clean_dataset(train_v1)

**Tokenize question and context together with special tokens.**
*   CPU time grows fast with sequence length, and a stride/window lets you keep contexts smaller while still preserving answers near boundaries.

In [14]:
def tokenize(sample_question, sample_context):
  document_stride = 128
  encoding_strided = tokenizer(
      sample_question,
      sample_context,
      truncation="only_second",
      max_length=512,
      stride=document_stride,
      return_overflowing_tokens=True,
      padding="max_length",
      return_offsets_mapping=True,
      return_tensors="pt"
  )
  return encoding_strided



In [15]:

sample_question = train_v1["data"][0]["paragraphs"][0]["qas"][0]["question"]
sample_context  = train_v1["data"][0]["paragraphs"][0]["context"]
encoding_strided = tokenize(sample_question, sample_context)

print(f"Number of chunks produced : {encoding_strided['input_ids'].shape[0]}")
print(f"Each chunk shape          : {encoding_strided['input_ids'].shape[1:]}")

for i in range(encoding_strided['input_ids'].shape[0]):
    tokens = encoding_strided['input_ids'][i].tolist()
    mask = encoding_strided['attention_mask'][i].tolist()
    real_tokens = mask.count(1)
    pad_tokens = mask.count(0)
    print(f"\nChunk {i+1}:")
    print(f"  Real tokens : {real_tokens}")
    print(f"  PAD tokens  : {pad_tokens}")
    print(f"  First 10 decoded: {tokenizer.convert_ids_to_tokens(tokens[:10])}")
    print(f"  Last  10 decoded: {tokenizer.convert_ids_to_tokens(tokens[-10:])}")

Number of chunks produced : 1
Each chunk shape          : torch.Size([512])

Chunk 1:
  Real tokens : 176
  PAD tokens  : 336
  First 10 decoded: ['[CLS]', 'to', 'whom', 'did', 'the', 'virgin', 'mary', 'allegedly', 'appear', 'in']
  Last  10 decoded: ['[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


**Compute answer start and end offsets**

In [16]:
def offset_mapping(answer, encoding_strided):
    ans_start = answer["answer_start"]
    ans_end   = ans_start + len(answer["text"])

    results = []

    for i in range(encoding_strided['input_ids'].shape[0]):
        offsets = encoding_strided['offset_mapping'][i].tolist()
        sequence_ids = encoding_strided.sequence_ids(i)  # tells us which tokens belong to the question (0) and which to the context (1)

        # Find where the context tokens start and end in this chunk
        ctx_start = next(j for j, s in enumerate(sequence_ids) if s == 1)
        ctx_end = len(sequence_ids) - 1 - next(j for j, s in enumerate(reversed(sequence_ids)) if s == 1)

        # Check if the answer is even inside this chunk
        if offsets[ctx_start][0] > ans_start or offsets[ctx_end][1] < ans_end:
            start_pos, end_pos = 0, 0
        else:
            # Walk tokens to find which one contains ans_start and ans_end
            start_pos = next((j for j in range(ctx_start, ctx_end+1) if offsets[j][0] <= ans_start < offsets[j][1]), ctx_start)
            end_pos = next((j for j in range(ctx_end, ctx_start-1, -1) if offsets[j][1] >= ans_end > offsets[j][0]), ctx_end)

        results.append((start_pos, end_pos))

    return results

In [17]:
sample_question = train_v1["data"][0]["paragraphs"][0]["qas"][0]["question"]
sample_context = train_v1["data"][0]["paragraphs"][0]["context"]
sample_answer = train_v1["data"][0]["paragraphs"][0]["qas"][0]["answers"][0]
encoding_strided = tokenize(sample_question, sample_context)
chunk_positions  = offset_mapping(sample_answer, encoding_strided)

print(f"Question : {sample_question}")
print(f"Answer   : {sample_answer}")
print(f"Context  : {sample_context[:150]}...")
print(f"Chunks   : {len(chunk_positions)}")

for i, (start_pos, end_pos) in enumerate(chunk_positions):
    print(f"\nChunk {i+1}: start_position={start_pos}, end_position={end_pos}")
    tokens = encoding_strided['input_ids'][i].tolist()
    if start_pos != 0 and end_pos != 0:
        predicted_answer = tokenizer.convert_tokens_to_string(
            tokenizer.convert_ids_to_tokens(tokens[start_pos:end_pos+1])
        )
        print(f"  Recovered answer: '{predicted_answer}'")
    else:
        print(f"  Answer not in this chunk")

Question : To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Answer   : {'answer_start': 515, 'text': 'Saint Bernadette Soubirous'}
Context  : Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front o...
Chunks   : 1

Chunk 1: start_position=130, end_position=137
  Recovered answer: 'saint bernadette soubirous'


*Note: Runnig Time 4-5 minuites*

**Create train/validation/test splits**
*   Split train_v1 into 80/10/10 Splits.
*   Subset Filterin, slice train set down to 50k QA pairs.

In [18]:
import random

# Fix seed for reproducibility
random.seed(42)

# Get all articles and shuffle
articles = train_v1["data"].copy()
random.shuffle(articles)

# Split at article level 80/10/10
total = len(articles)
train_end = int(0.8 * total)
val_end = int(0.9 * total)

train_articles = articles[:train_end]
val_articles = articles[train_end:val_end]
test_articles = articles[val_end:]

print(f"Train articles      : {len(train_articles)}")
print(f"Validation articles : {len(val_articles)}")
print(f"Test articles       : {len(test_articles)}")

Train articles      : 353
Validation articles : 44
Test articles       : 45


In [19]:
subset_articles = []
qa_count = 0

for article in train_articles:
    article_qa = sum(len(para["qas"]) for para in article["paragraphs"])
    if qa_count <= 50000:
        subset_articles.append(article)
        qa_count += article_qa
    if qa_count >= 50000:
        break

train_articles = subset_articles
print(f"Subset articles : {len(train_articles)}")
print(f"Subset QA pairs : {qa_count}")

Subset articles : 245
Subset QA pairs : 50012


In [20]:
for article in train_articles:
    for para in article["paragraphs"]:
        for qa in para["qas"]:
            encoding_strided = tokenize(qa["question"], para["context"])
            answer = qa["answers"][0]
            chunk_positions = offset_mapping(answer, encoding_strided)

            for i, (start_pos, end_pos) in enumerate(chunk_positions):
                train_encoding_strided.append(encoding_strided)
                train_input_ids.append(encoding_strided['input_ids'][i].tolist())
                train_attention_masks.append(encoding_strided['attention_mask'][i].tolist())
                train_start_positions.append(start_pos)
                train_end_positions.append(end_pos)

In [21]:
for article in val_articles:
    for para in article["paragraphs"]:
        for qa in para["qas"]:
            encoding_strided = tokenize(qa["question"], para["context"])
            answer = qa["answers"][0]
            chunk_positions  = offset_mapping(answer, encoding_strided)

            for i, (start_pos, end_pos) in enumerate(chunk_positions):
                val_encoding_strided.append(encoding_strided)
                val_input_ids.append(encoding_strided['input_ids'][i].tolist())
                val_attention_masks.append(encoding_strided['attention_mask'][i].tolist())
                val_start_positions.append(start_pos)
                val_end_positions.append(end_pos)

In [22]:
for article in test_articles:
    for para in article["paragraphs"]:
        for qa in para["qas"]:
            encoding_strided = tokenize(qa["question"], para["context"])
            answer = qa["answers"][0]
            chunk_positions  = offset_mapping(answer, encoding_strided)

            for i, (start_pos, end_pos) in enumerate(chunk_positions):
                test_encoding_strided.append(encoding_strided)
                test_input_ids.append(encoding_strided['input_ids'][i].tolist())
                test_attention_masks.append(encoding_strided['attention_mask'][i].tolist())
                test_start_positions.append(start_pos)
                test_end_positions.append(end_pos)

In [23]:
print(f"Train chunks      : {len(train_input_ids)}")
print(f"Validation chunks : {len(val_input_ids)}")
print(f"Test chunks       : {len(test_input_ids)}")

Train chunks      : 50068
Validation chunks : 9308
Test chunks       : 8078


In [24]:
def count_qa(articles):
    return sum(
        len(para["qas"])
        for article in articles
        for para in article["paragraphs"]
    )

print(f"Train QA pairs      : {count_qa(train_articles)}")
print(f"Validation QA pairs : {count_qa(val_articles)}")
print(f"Test QA pairs       : {count_qa(test_articles)}")

Train QA pairs      : 50012
Validation QA pairs : 9303
Test QA pairs       : 8073


## Computing baseline EM and F1 scores using the official SQuAD evaluation script



In [25]:
baseline_predictions = {}
em_scores = []
f1_scores = []

for article in val_articles:
    for para in article["paragraphs"]:
        context  = para["context"]
        for qa in para["qas"]:
            question = qa["question"]
            gold_answers = [a["text"] for a in qa["answers"]]

            sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', context) if s.strip()]
            if len(sentences) == 0:
                prediction = ""
            elif len(sentences) == 1:
                prediction = sentences[0]
            else:
                tokenized_sentences = [s.lower().split() for s in sentences]
                tokenized_question  = question.lower().split()
                bm25 = BM25Okapi(tokenized_sentences)
                scores = bm25.get_scores(tokenized_question)
                prediction = sentences[np.argmax(scores)]

            baseline_predictions[qa["id"]] = prediction
            em_scores.append(max(exact_match_score(a, prediction) for a in gold_answers))
            f1_scores.append(max(f1_score(a, prediction) for a in gold_answers))

avg_em = sum(em_scores) / len(em_scores) * 100
avg_f1 = sum(f1_scores) / len(f1_scores) * 100

print(f"BM25 Baseline Results on Validation Set")
print(f"Total examples : {len(em_scores)}")
print(f"Exact Match    : {avg_em:.2f}%")
print(f"F1 Score       : {avg_f1:.2f}%")

BM25 Baseline Results on Validation Set
Total examples : 9303
Exact Match    : 0.12%
F1 Score       : 14.19%


In [26]:
sample_para = val_articles[0]["paragraphs"][0]
sample_qa   = sample_para["qas"][0]
question    = sample_qa["question"]
context     = sample_para["context"]
gold        = [a["text"] for a in sample_qa["answers"]]

sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', context) if s.strip()]
tokenized_sentences = [s.lower().split() for s in sentences]
tokenized_question  = question.lower().split()
bm25 = BM25Okapi(tokenized_sentences)
scores = bm25.get_scores(tokenized_question)
prediction = sentences[np.argmax(scores)]

print(f"Question    : {question}")
print(f"Gold answers: {gold}")
print(f"Prediction  : {prediction}")
print(f"EM          : {max(exact_match_score(a, prediction) for a in gold)}")
print(f"F1          : {max(f1_score(a, prediction) for a in gold):.4f}")

Question    : Who founded Philadelphia?
Gold answers: ['William Penn']
Prediction  : In 1682, William Penn founded the city to serve as capital of the Pennsylvania Colony.
EM          : False
F1          : 0.2667


In [27]:
with open("bm25_baseline_predictions.json", "w") as f:
    json.dump(baseline_predictions, f)

### 🔷 <font color="#9CE2FF">Sherouk's Work Ends Here</font>

# Student B

In [28]:
import accelerate
print(accelerate.__version__)
print(accelerate.__file__)

0.30.1
/usr/local/lib/python3.12/dist-packages/accelerate/__init__.py


## 🏗️ 4. Model Architecture — DistilBertForQuestionAnswering

In [29]:
import os, time, tracemalloc, random, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)

warnings.filterwarnings("ignore")

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Hardware ─────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ── Hyperparameters ───────────────────────────────────────────────────────────
MODEL_CHECKPOINT  = "distilbert-base-uncased"
MAX_SEQ_LENGTH    = 384
STRIDE            = 128
BATCH_SIZE        = 16
GRAD_ACCUM_STEPS  = 2
NUM_EPOCHS        = 3
LEARNING_RATE     = 2e-5
WEIGHT_DECAY      = 0.01
OUTPUT_DIR        = "./distilbert-squad-output"

print("Configuration loaded ✅")

2026-06-11 06:08:37.787923: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781158117.812223     546 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781158117.819979     546 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781158117.839271     546 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781158117.839303     546 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781158117.839306     546 computation_placer.cc:177] computation placer alr

Using device: cuda
Configuration loaded ✅


# load data

In [30]:
# BUG FIX 1: build HuggingFace Datasets from the lists produced by Student A
#             (val_dataset was never defined; Student B referenced a non-existent variable)
# BUG FIX 2: train_input_ids is a plain list, not a Dataset — cannot call .set_format()
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "input_ids":       train_input_ids,
    "attention_mask":  train_attention_masks,
    "start_positions": train_start_positions,
    "end_positions":   train_end_positions,
})

val_dataset = Dataset.from_dict({
    "input_ids":       val_input_ids,
    "attention_mask":  val_attention_masks,
    "start_positions": val_start_positions,
    "end_positions":   val_end_positions,
})

train_dataset.set_format("torch")
val_dataset.set_format("torch")

print(f"Train dataset : {len(train_dataset):,} chunks")
print(f"Val dataset   : {len(val_dataset):,} chunks")

Train dataset : 50,068 chunks
Val dataset   : 9,308 chunks


## 🏗️ 4. Model Architecture — DistilBertForQuestionAnswering

In [31]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DistilBertForQuestionAnswering.from_pretrained(MODEL_CHECKPOINT).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model          : {MODEL_CHECKPOINT}")
print(f"Total params   : {total_params/1e6:.1f}M")
print(f"Trainable      : {trainable_params/1e6:.1f}M")
print(f"Layers         : 6 Transformer encoder layers")
print(f"Hidden size    : 768  |  Heads: 12  |  FFN: 3072")
print()
print(model)


Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model          : distilbert-base-uncased
Total params   : 66.4M
Trainable      : 66.4M
Layers         : 6 Transformer encoder layers
Hidden size    : 768  |  Heads: 12  |  FFN: 3072

DistilBertForQuestionAnswering(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bi

In [32]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla P100-PCIE-16GB


In [33]:
#train_dataset = train_dataset.select(range(10000))

## 🏋️ 5. Training + Loss Curves

In [ ]:


training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    learning_rate               = LEARNING_RATE,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    num_train_epochs            = NUM_EPOCHS,
    weight_decay                = WEIGHT_DECAY,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    seed                        = SEED,
    logging_steps               = 50,
    report_to                   = "none",
    fp16=True
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    data_collator = default_data_collator,
)

print("Starting training...")
train_start  = time.time()
train_output = trainer.train()
train_end    = time.time()
wall_clock_hrs = (train_end - train_start) / 3600
print(f"✅ Training complete — wall-clock time: {wall_clock_hrs:.2f} hours")

# ── Loss Curves ────────────────────────────────────────────────────────────────
log_history = trainer.state.log_history

train_steps  = [e["step"]      for e in log_history if "loss"      in e]
train_losses = [e["loss"]      for e in log_history if "loss"      in e]
eval_epochs  = [e["epoch"]     for e in log_history if "eval_loss" in e]
eval_losses  = [e["eval_loss"] for e in log_history if "eval_loss" in e]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_steps, train_losses, label="Train loss", color="#4C72B0", linewidth=1.2)
ax2 = ax.twinx()
total_train_steps  = train_steps[-1] if train_steps else 1
eval_steps_approx  = [int(e / NUM_EPOCHS * total_train_steps) for e in eval_epochs]
ax2.plot(eval_steps_approx, eval_losses, "o--", label="Val loss", color="#DD8452", linewidth=1.5)

ax.set_xlabel("Global step")
ax.set_ylabel("Train loss")
ax2.set_ylabel("Val loss")
ax.set_title("Training & Validation Loss Curves", fontsize=13, fontweight="bold")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
plt.tight_layout()
plt.savefig("loss_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("Loss curves saved → loss_curves.png")

Starting training...


Epoch,Training Loss,Validation Loss
1,1.434200,1.199479
2,1.133600,1.095660


In [ ]:
SAVE_PATH = "./distilbert-squad-final"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model and tokenizer saved → {SAVE_PATH}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

SAVE_PATH = "./distilbert-squad-final"
model = AutoModelForQuestionAnswering.from_pretrained(SAVE_PATH)
tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
model.eval()

def qa_predict(question, context):
    inputs = tokenizer(question, context, return_tensors="pt", truncation=True, max_length=384)
    with torch.no_grad():
        outputs = model(**inputs)
    start = outputs.start_logits.argmax().item()
    end = outputs.end_logits.argmax().item() + 1
    return tokenizer.decode(inputs["input_ids"][0][start:end], skip_special_tokens=True)

demos = [
    {"question": "Who introduced BERT?", "context": "BERT was introduced by Devlin et al. from Google AI Language in 2019."},
    {"question": "How many parameters does DistilBERT have?", "context": "DistilBERT reduces BERT's parameter count by 40%, resulting in approximately 66 million parameters."},
    {"question": "What dataset was used for evaluation?", "context": "The model was evaluated on SQuAD v1.1, which contains over 107,000 question-answer pairs."},
]

for d in demos:
    answer = qa_predict(d["question"], d["context"])
    print(f"Q: {d['question']}")
    print(f"A: {answer}")
    print()

In [ ]:
import time
import torch

# 1. تحديد الجهاز كـ CPU
device_cpu = torch.device("cpu")

# نقل الموديل للـ CPU
model.to(device_cpu)
model.eval()

# 2. تجهيز سؤال ونص تجريبي

sample_para = val_articles[0]["paragraphs"][0]
sample_qa   = sample_para["qas"][0]
question    = sample_qa["question"]
context     = sample_para["context"]

# 3. Tokenization
inputs = tokenizer(
    question,
    context,
    max_length=384,
    truncation="only_second",
    return_tensors="pt"
)

# نقل الـ tensors للـ CPU
inputs = {k: v.to(device_cpu) for k, v in inputs.items()}

# 4. قياس زمن الـ inference
start_time = time.time()

with torch.no_grad():
    outputs = model(**inputs)

end_time = time.time()

cpu_latency = (end_time - start_time) * 1000

print(f"✅ Question: {question}")
print(f"🚀 Inference time on CPU: {cpu_latency:.2f} ms")

# لو هتكمل تدريب/تجارب على GPU
# model.to("cuda")

### 🔷 <font color="#9CE2FF">Sherouk's Work:Baseline Distilbert Optimization</font>

# Optimization plan for the initial model

Before running any ablation studies, I will first optimize the base model in a structured way. The goal is to make sure the model is stable, trainable, and reasonably tuned before testing any design choices.

## Phase 1: Sanity check / overfitting test
I will first train the model on a very small subset of the data, around 100 – 200 examples, for several epochs. This step is only to confirm that the pipeline works correctly and that the model can overfit a tiny dataset. If the training loss drops close to zero, it means the model, data loading, and training setup are functioning properly.

## Phase 2: Coarse hyperparameter search
After the sanity check, I will run a broad search over the most important training parameters, such as learning rate, number of epochs, and weight decay. The purpose of this phase is not to find the perfect setting immediately, but to identify a strong range of values that performs well.

## Phase 3: Fine hyperparameter search
Once the best region is identified, I will narrow the search space and test more precise values around the strongest configuration. This step is intended to refine the initial model and improve performance before moving to more experimental settings.

Overall, this workflow ensures that the model is first validated, then tuned, and only after that used for ablation experiments. This makes the final results more reliable and easier to interpret.


In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import DistilBertForQuestionAnswering, pipeline

SAVE_PATH = "./distilbert-squad-final"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Choose small data subset to force the model to overfitting
overfit_train = train_dataset.select(range(150))
overfit_val = val_dataset.select(range(30))

# Hyperparameters Initialization
OVERFIT_EPOCHS = 50
OVERFIT_LR = 2e-4   # high lr on purpose to force overfitting
OVERFIT_BATCH  = 16

# Load the baseline destilbert model 
overfit_model = DistilBertForQuestionAnswering.from_pretrained(SAVE_PATH).to(DEVICE)
optimizer = AdamW(overfit_model.parameters(), lr=OVERFIT_LR)

train_loader = DataLoader(overfit_train, batch_size=OVERFIT_BATCH, shuffle=True)
val_loader = DataLoader(overfit_val, batch_size=OVERFIT_BATCH, shuffle=False)

print(f"Overfit check: {len(overfit_train)} train / {len(overfit_val)} val examples")
print(f"Epochs: {OVERFIT_EPOCHS}  |  LR: {OVERFIT_LR}  |  Batch: {OVERFIT_BATCH}")
print("=" * 75)

# Training
for epoch in range(1, OVERFIT_EPOCHS + 1):
    overfit_model.train()
    train_loss  = 0.0
    train_steps = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        start_positions = batch["start_positions"].to(DEVICE)
        end_positions = batch["end_positions"].to(DEVICE)

        optimizer.zero_grad()
        outputs = overfit_model(
            input_ids = input_ids,
            attention_mask = attention_mask,
            start_positions = start_positions,
            end_positions = end_positions
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_steps += 1

    avg_train_loss = train_loss / train_steps

    # Validation
    overfit_model.eval()
    val_loss  = 0.0
    val_steps = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            start_positions = batch["start_positions"].to(DEVICE)
            end_positions = batch["end_positions"].to(DEVICE)

            outputs = overfit_model(
                input_ids = input_ids,
                attention_mask = attention_mask,
                start_positions = start_positions,
                end_positions = end_positions
            )
            val_loss += outputs.loss.item()
            val_steps += 1

    avg_val_loss = val_loss / val_steps

    print(
        f"Finished epoch {epoch:>2} / {OVERFIT_EPOCHS}: "
        f"train loss: {avg_train_loss:.6f},  "
        f"val loss: {avg_val_loss:.6f},  "
        f"lr {OVERFIT_LR:.2e}"
    )

print("=" * 80)
print(f"Finished optimization. Final train loss: {avg_train_loss:.6f}  |  val loss: {avg_val_loss:.6f}")

# Cleanup
del overfit_model, optimizer, train_loader, val_loader
torch.cuda.empty_cache()

## Phase 1: Overfitting Sanity Check

I implemented the overfitting step from Phase 1 by training the model on **150 samples** and using **30 validation samples** for **50 epochs**.

- **Train loss** dropped from **2.92** to **0.19** by **epoch 10**.
- The lowest value for the **Train loss** is **0.06** at epoce **28**.
- **Validation loss** increased while training loss kept falling, which shows that the model is overfitting the small subset.
- After **epoch 28**, the training loss started bouncing back up, likely because of the large number of epochs, but this does not matter here because the goal was only to confirm that overfitting is achievable.

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import itertools

# Candidates
LR_CANDIDATES = [3e-5, 2e-5, 1e-5]
EPOCH_CANDIDATES = [2, 3]
WARMUP_CANDIDATES = [0.0, 0.1]
WEIGHT_DECAY_CANDIDATES = [0.0, 0.01]
BATCH_SIZE = 16          # keep fixed during coarse search

coarse_train = train_dataset.select(range(int(len(train_dataset) * 0.10))) # 10 % Of training and validation data set
coarse_val = val_dataset.select(range(int(len(val_dataset) * 0.10)))

# Data loaders (full dataset this time)
coarse_train_loader = DataLoader(coarse_train, batch_size=BATCH_SIZE, shuffle=True)
coarse_val_loader = DataLoader(coarse_val, batch_size=BATCH_SIZE, shuffle=False)

results = []   # will hold (lr, epochs, warmup, final_val_loss)

# Coarse search
for lr, epochs, warmup_ratio, weight_decay in itertools.product(LR_CANDIDATES, EPOCH_CANDIDATES, WARMUP_CANDIDATES, WEIGHT_DECAY_CANDIDATES):
    print(f"\n{'='*75}")
    print(f"  Run: lr = {lr:.0e}  |  epochs = {epochs}  |  warmup = {warmup_ratio}")
    print(f"{'='*75}")

    model = DistilBertForQuestionAnswering.from_pretrained(SAVE_PATH).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_steps = len(coarse_train_loader) * epochs
    warmup_steps = int(total_steps * warmup_ratio)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps = warmup_steps,
        num_training_steps = total_steps
    )

    for epoch in range(1, epochs + 1):
        # Training
        model.train()
        train_loss, train_steps = 0.0, 0

        for batch in coarse_train_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            start_positions = batch["start_positions"].to(DEVICE)
            end_positions = batch["end_positions"].to(DEVICE)

            optimizer.zero_grad()
            outputs = model(
                input_ids = input_ids,
                attention_mask = attention_mask,
                start_positions = start_positions,
                end_positions = end_positions
            )
            outputs.loss.backward()

            # Gradient clipping in order to prevent the exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()

            train_loss += outputs.loss.item()
            train_steps += 1

        avg_train_loss = train_loss / train_steps

        # Validatation
        model.eval()
        val_loss, val_steps = 0.0, 0

        with torch.no_grad():
            for batch in coarse_val_loader:
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                start_positions = batch["start_positions"].to(DEVICE)
                end_positions = batch["end_positions"].to(DEVICE)

                outputs = model(
                    input_ids = input_ids,
                    attention_mask  = attention_mask,
                    start_positions = start_positions,
                    end_positions   = end_positions
                )
                val_loss += outputs.loss.item()
                val_steps += 1

        avg_val_loss = val_loss / val_steps

        current_lr = scheduler.get_last_lr()[0]
        print(
            f"  epoch {epoch}/{epochs}   "
            f"train loss: {avg_train_loss:.4f}  |  "
            f"val loss: {avg_val_loss:.4f}  |  "
            f"lr: {current_lr:.2e}"
        )

        # Early explosion check where if training loss > 3x original skip
        if epoch == 1 and avg_train_loss > 3 * avg_train_loss:
            print("Loss exploded skip this combo")
            break

    results.append({
        "lr": lr,
        "epochs": epochs,
        "warmup": warmup_ratio,
        "weight_decay": weight_decay,
        "val_loss": avg_val_loss
    })
    del model, optimizer, scheduler

print(f"\n{'='*80}")
print("Coarse search results (sorted by validation loss):")
print(f"{'='*80}")
for r in sorted(results, key=lambda x: x["val_loss"]):
    print(f"  lr={r['lr']:.0e}  epochs={r['epochs']}  warmup={r['warmup']}  weight_decay={r['weight_decay']}  =>  val_loss={r['val_loss']:.4f}")

coarse_results_df = pd.DataFrame(sorted(results, key=lambda x: x["val_loss"]))
coarse_results_df.to_csv("coarse_search_results.csv", index=False)

print("\nCoarse search results saved into file 'coarse_search_results.csv'")
print(coarse_results_df.to_string(index=False))

### Coarse Search Space Justification

Based on the coarse grid search results, we selected the following values for the fine search space:

- **Learning rate = 1e-05**
- **Number of epochs = 2**
- **Warmup ratio = 0.1**
- **Weight decay = 0.01**

These values were chosen because they produced the **lowest validation loss** among all tested configurations in the coarse search.

#### Why this choice is reasonable
- **Learning rate:** `1e-05` consistently outperformed `2e-05` and `3e-05`, suggesting that a smaller update step led to more stable optimization and better generalization.
- **Epochs:** `2` epochs gave better validation performance than `3` epochs, indicating that training longer likely started to overfit.
- **Warmup:** `0.1` slightly improved results in the best configuration, so it was kept as the preferred value for refinement.
- **Weight decay:** `0.01` gave the best validation loss overall, showing that a small amount of regularization helped the model generalize better.

#### Conclusion
Since the coarse search already identified a clear best region, the fine search is centered around the most promising hyperparameters instead of exploring the full space again. This makes the next stage of tuning more efficient while preserving the best generalization performance observed so far.

In [ ]:
FINE_LR_CANDIDATES = [1e-5, 1.5e-5, 2e-5]
FINE_EPOCH_CANDIDATES = [2, 3]
FINE_WARMUP_CANDIDATES = [0.06, 0.1]
FINE_WEIGHT_DECAY_CANDIDATES = [0.005, 0.01]

BATCH_SIZE = 16

fine_train = train_dataset.select(range(int(len(train_dataset) * 0.20)))
fine_val = val_dataset.select(range(int(len(val_dataset) * 0.20)))

fine_train_loader = DataLoader(fine_train, batch_size=BATCH_SIZE, shuffle=True)
fine_val_loader = DataLoader(fine_val, batch_size=BATCH_SIZE, shuffle=False)

print(f"Fine search: {len(fine_train)} train / {len(fine_val)} val examples")
print(f"Total combinations: {len(FINE_LR_CANDIDATES) * len(FINE_EPOCH_CANDIDATES) * len(FINE_WARMUP_CANDIDATES) * len(FINE_WEIGHT_DECAY_CANDIDATES)}")

fine_results = []

for lr, epochs, warmup_ratio, weight_decay in itertools.product(
    FINE_LR_CANDIDATES,
    FINE_EPOCH_CANDIDATES,
    FINE_WARMUP_CANDIDATES,
    FINE_WEIGHT_DECAY_CANDIDATES
):
    print(f"\n{'='*75}")
    print(f"  Run: lr={lr:.1e}  |  epochs={epochs}  |  warmup={warmup_ratio}  |  wd={weight_decay}")
    print(f"{'='*75}")

    model = DistilBertForQuestionAnswering.from_pretrained(SAVE_PATH).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_steps = len(fine_train_loader) * epochs
    warmup_steps = int(total_steps * warmup_ratio)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = warmup_steps,
        num_training_steps = total_steps
    )

    best_val_loss = float("inf")

    for epoch in range(1, epochs + 1):
        # Training
        model.train()
        train_loss, train_steps = 0.0, 0

        for batch in fine_train_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            start_positions = batch["start_positions"].to(DEVICE)
            end_positions = batch["end_positions"].to(DEVICE)

            optimizer.zero_grad()
            outputs = model(
                input_ids = input_ids,
                attention_mask = attention_mask,
                start_positions = start_positions,
                end_positions = end_positions
            )
            outputs.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            train_loss += outputs.loss.item()
            train_steps += 1

        avg_train_loss = train_loss / train_steps

        # Validation
        model.eval()
        val_loss, val_steps = 0.0, 0

        with torch.no_grad():
            for batch in fine_val_loader:
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                start_positions = batch["start_positions"].to(DEVICE)
                end_positions = batch["end_positions"].to(DEVICE)

                outputs = model(
                    input_ids = input_ids,
                    attention_mask = attention_mask,
                    start_positions = start_positions,
                    end_positions = end_positions
                )
                val_loss += outputs.loss.item()
                val_steps += 1

        avg_val_loss = val_loss / val_steps

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss

        current_lr = scheduler.get_last_lr()[0]
        print(
            f"  epoch {epoch}/{epochs}   "
            f"train loss: {avg_train_loss:.4f}  |  "
            f"val loss: {avg_val_loss:.4f}  |  "
            f"lr: {current_lr:.2e}"
        )

    fine_results.append({
        "lr": lr,
        "epochs": epochs,
        "warmup": warmup_ratio,
        "weight_decay": weight_decay,
        "val_loss": best_val_loss
    })

    del model, optimizer, scheduler
    torch.cuda.empty_cache()

print(f"\n{'='*80}")
print("Fine Search Results (sorted by validation loss):")
print(f"{'='*80}")
for r in sorted(fine_results, key=lambda x: x["val_loss"]):
    print(
        f"  lr={r['lr']:.1e}  epochs={r['epochs']}  "
        f"warmup={r['warmup']}  wd={r['weight_decay']}  "
        f"→  val_loss={r['val_loss']:.4f}"
    )

fine_results_df = pd.DataFrame(sorted(fine_results, key=lambda x: x["val_loss"]))
fine_results_df.to_csv("fine_search_results.csv", index=False)

print("Fine search results saved into file 'fine_search_results.csv'")
print(fine_results_df.to_string(index=False))

In [ ]:
best_coarse = coarse_results_df.iloc[0]
best_lr = best_coarse["lr"]
best_wd = best_coarse["weight_decay"]
best_warmup = best_coarse["warmup"]
best_epoch = int(best_coarse["epochs"])


In [ ]:
print(f"\nFine search space:")
print(f"  LR            : {best_lr}")
print(f"  Epoch         : {best_epoch}")
print(f"  Warmup        : {best_warmup}")
print(f"  Weight decay  : {best_wd}")


Early stopping was used to reduce overfitting and select the checkpoint with the best validation loss. Since training loss continued to decrease while validation loss began to increase in some runs, early stopping helped preserve generalization and avoided unnecessary training epochs.

### 🔷 <font color="#9CE2FF">Sherouk's Work: Distilbert Ablations</font>

## Upcoming Ablation Plan

For the next phase, I will run two ablation families on the DistilBERT QA model:

### 1) Layer-Freezing Ablation
My goal is to measure:
1.   **Task Adaptation vs. Knowledge Retention:**<br> Balancing how much the model learns a new specific task versus how much it "forgets" its original pre-trained knowledge.
2.  **Computational Efficiency:**<br> Measuring the reduction in training time per epoch, total memory usage, and the number of floating-point operations (FLOPs) required to update only a fraction of the network.

Planned settings:
- **Full fine-tuning**: no layers frozen.
- **Freeze embeddings only**.
- **Freeze embeddings + bottom 2 transformer layers**.
- **Freeze embeddings + bottom 4 transformer layers**.

This keeps the ablation **partial**, not full freezing, because the model still needs trainable upper layers to adapt to extractive QA as those layer store the common features.


### Why these ablations
The layer-freezing idea is supported by prior work showing that fine-tuning often changes only part of the model, and that partial freezing can preserve performance while reducing cost *[1]*.  
A more recent study also reports that freezing a subset of lower transformer layers can improve training efficiency and preserve or improve task performance *[3]*.  
A transfer-learning study further supports the idea that freezing, reinitialization, and truncation are valid ways to study layer transferability and optimal cut points *[2]*.

### How I will evaluate
For each setting, I will compare:
- **Exact Match (EM)**
- **F1 Score**
- **Training Time**
- **Inference Latency**
- **Number of Trainable Parameters**

### References
[1] https://aclanthology.org/2020.blackboxnlp-1.4.pdf  
[2] https://www.ifi.uzh.ch/dam/jcr:76acccb6-0dfd-4b5c-9df8-ee049133f9f9/fazla_guzman_manekar2024master_project.pdf  
[3] https://openreview.net/pdf?id=kvBuxFxSLR

**1. Define function `apply_freeze` this function is the main drive of layer freezing as it will freez layer based on specific configuration e.g., whether to freeze embedding layers or not.**

In [ ]:
def apply_freeze(model, freeze_embeddings, freeze_layers):
    # Start with everything trainable
    for param in model.parameters():
        param.requires_grad = True

    # Freeze embeddings if requested
    if freeze_embeddings:
        for name, param in model.named_parameters():
            if "distilbert.embeddings" in name:
                param.requires_grad = False

    # Freeze specified transformer layers
    for layer_idx in freeze_layers:
        for name, param in model.named_parameters():
            if f"distilbert.transformer.layer.{layer_idx}." in name:
                param.requires_grad = False

**2. Define function `count_trainable` which is responsible for counting the models's total params, trainable params, and frozen params.** (These numbers are going to be used for the eveluation of each ablation)

In [ ]:
def count_trainable(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable

    print(f"Total params    : {total/1e6:.2f}M")
    print(f"Trainable params: {trainable/1e6:.2f}M")
    print(f"Frozen params   : {frozen/1e6:.2f}M")

    return total, trainable, frozen

**4. Define the 4 layer freezing ablations that we would study.**

In [ ]:
ABLATION_CONFIGS = {
    "full_finetune": {
        "freeze_embeddings": False,
        "freeze_layers":     []
    },
    "emb_2layers": {
        "freeze_embeddings": True,
        "freeze_layers":     [0, 1]
    },
    "emb_4layers": {
        "freeze_embeddings": True,
        "freeze_layers":     [0, 1, 2, 3]
    },
}

print("Ablation configs defined:")
for name, cfg in ABLATION_CONFIGS.items():
    print(f"  {name}: freeze_embeddings={cfg['freeze_embeddings']}, freeze_layers={cfg['freeze_layers']}")

**5. Loop over the 4 freeze ablation and for each one:**<br>

*  Load the checkpoint model from Hugging Face, this would be the initial version of the model.
*  Apply the freezing.
*  Retrain the model after freezing and produce a new version.
*  Count the trainable and the frozen parameters.
*  Record training time per epoch.
*  Measure the inference letancy.
*  Evaluate using Exact Match & F1 Score.
*  Save the results of all the metrics.




In [ ]:
latency_sample = val_examples[0]  # A single sample would be used for letancy evaluation.
ablation_results = {}
# After (optimal from coarse search)
LEARNING_RATE = 1e-5
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
NUM_EPOCHS = 2
MAX_SEQ_LENGTH = 384
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
SEED = 42

for config_name, config in ABLATION_CONFIGS.items():
    print(f"\n{'='*55}")
    print(f"Config: {config_name}")
    print(f"{'='*55}")

    # Load the checkpoint model from Hugging Face.
    ablation_model = DistilBertForQuestionAnswering.from_pretrained(
        "distilbert-base-uncased"
    ).to(DEVICE)
    # Apply the freezing.
    apply_freeze(
        ablation_model,
        freeze_embeddings=config["freeze_embeddings"],
        freeze_layers=config["freeze_layers"]
    )

    # Count the trainable and the frozen parameters.
    total, trainable, frozen = count_trainable(ablation_model)

    # Retrain the model after freezing and produce a new version.
    ablation_args = TrainingArguments(
        output_dir = f"./ablation_{config_name}",
        eval_strategy = "epoch",
        save_strategy = "epoch",
        learning_rate = LEARNING_RATE,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM_STEPS,
        num_train_epochs = NUM_EPOCHS,
        weight_decay = WEIGHT_DECAY,
        warmup_ratio = WARMUP_RATIO,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        seed = SEED,
        logging_steps = 50,
        report_to = "none",
        fp16 = True,
    )

    ablation_trainer = Trainer(
        model = ablation_model,
        args = ablation_args,
        train_dataset = train_dataset,
        eval_dataset = val_dataset,
        data_collator = default_data_collator,
    )

    # Record the total training time
    training_time_starts_at = time.time()
    ablation_trainer.train()
    train_time_hrs = (time.time() - training_time_starts_at) / 3600
    print(f"Total Training time: {train_time_hrs:.2f} hrs")

    # get the per epoch time from log history
    log_history = ablation_trainer.state.log_history
    eval_entries = [e for e in log_history if "eval_loss" in e]
    per_epoch_time_sec = (time.time() - training_time_starts_at) / NUM_EPOCHS

    print(f"Avg time per epoch   : {per_epoch_time_sec:.2f} sec")

    # Measure the Inference Letancy
    ablation_model.eval()
    inputs = tokenizer(
        latency_sample["question"],
        latency_sample["context"],
        max_length=384,
        truncation="only_second",
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        # Warm-up run (not measured)
        _ = ablation_model(**inputs)

        t_lat = time.time()
        _ = ablation_model(**inputs)
        latency_ms = (time.time() - t_lat) * 1000

    print(f"Inference latency: {latency_ms:.2f} ms")

    # Evaluation using Exact Match and F1 Score
    ablation_pipeline = pipeline(
        "question-answering",
        model=ablation_model,
        tokenizer=tokenizer,
        device=0 if torch.cuda.is_available() else -1
    )

    em_scores, f1_scores = [], []
    for ex in val_examples:
        output = ablation_pipeline(
            question=ex["question"],
            context=ex["context"],
            max_answer_len=40
        )
        em, f1 = score_prediction(ex["gold_answers"], output["answer"])
        em_scores.append(em)
        f1_scores.append(f1)

    config_em = np.mean(em_scores) * 100
    config_f1 = np.mean(f1_scores) * 100

    print(f"EM : {config_em:.2f}%")
    print(f"F1 : {config_f1:.2f}%")

    # Save Results
    ablation_results[config_name] = {
        "em": config_em,
        "f1": config_f1,
        "trainable_params": trainable,
        "frozen_params": frozen,
        "train_time_hrs": train_time_hrs,
        "avg_epoch_time_sec": per_epoch_time_sec,
        "latency_ms": latency_ms,
    }

    # Free GPU memory before next config
    del ablation_model, ablation_trainer, ablation_pipeline
    torch.cuda.empty_cache()

print("\n✅ Layer freezing ablation complete")

**6. Build a summary DataFrame with columns: Config, EM, F1, Trainable Params, Training Time, Inference Latency.**

In [ ]:
summary_rows = []

for config_name, metrics in ablation_results.items():
    summary_rows.append({
        "Config": config_name,
        "EM (%)": round(metrics["em"], 2),
        "F1 (%)": round(metrics["f1"], 2),
        "Trainable Params (M)": round(metrics["trainable_params"] / 1e6, 2),
        "Frozen Params (M)": round(metrics["frozen_params"] / 1e6, 2),
        "Total Train Time (hrs)": round(metrics["train_time_hrs"], 2) if metrics["train_time_hrs"] else None,
        "Avg Epoch Time (sec)": round(metrics["avg_epoch_time_sec"], 2) if metrics["avg_epoch_time_sec"] else None,
        "Inference Latency (ms)": round(metrics["latency_ms"], 2) if metrics["latency_ms"] else None,
    })

ablation_summary_df = pd.DataFrame(summary_rows)
ablation_summary_df.to_csv("ablation_layer_freezing.csv", index=False)

print(ablation_summary_df.to_string(index=False))
print("\nSaved into file 'ablation_layer_freezing.csv'")

## Evaluation Plan

In the evaluation phase, I will compare the four DistilBERT ablation settings using both **quality metrics** and **efficiency metrics**.  
The main goal is to identify the best trade-off between answer quality and resource cost, rather than only the highest score on a single metric.


## Figures and Purpose

### Figure 1: EM and F1 grouped bar chart
This figure compares the answer quality of all ablation settings side by side.  
EM measures exact span correctness, while F1 measures overlap quality when the prediction is close but not exact.  
This chart is important because it shows whether freezing hurts QA performance and whether any frozen setup can match or nearly match full fine-tuning.

### Figure 2: Training time and latency grouped bar chart
This figure compares training cost and inference cost across the four settings.  
Training time shows the cost of optimization, while latency shows how fast the model runs at prediction time.  
This figure is important because an ablation is only useful if it improves or preserves quality while reducing computational cost.

### Figure 3: Trainable parameters bar chart
This figure shows how many parameters remain trainable under each freezing configuration.  
It makes the effect of freezing directly visible and helps explain why some settings train faster than others.  
This is important because parameter reduction is one of the main reasons for using layer freezing in the first place.

### Figure 4: F1 vs latency scatter plot with labels
This figure shows the trade-off between accuracy and deployment speed.  
Each point represents one ablation setting, with higher F1 and lower latency being preferable.  
This is important because it helps identify the best practical configuration, not just the best score on one metric.

### Figure 5: Radar chart
This figure gives a compact multi-metric view of all configurations.  
It will include normalized values for EM, F1, training time, latency, and trainable parameters so that different scales can be compared on the same plot.  
This is important for a final visual summary because it makes the strengths and weaknesses of each setting easy to see at a glance.

---

## How I will choose the best ablation
The best configuration will be the one that gives the strongest balance between:
- high EM and F1,
- low training time,
- low latency,
- and fewer trainable parameters.

---

## Related Work
This evaluation style is supported by prior work that studies **accuracy and efficiency** together in transformer models.  
A DistilBERT QA efficiency study also reported performance and latency under different settings, showing that QA models should be judged by both quality and cost [1].  

### References
  
[1] Improving QA Efficiency with DistilBERT: Fine-Tuning and Inference Analysis [https://arxiv.org/abs/2505.22937](https://arxiv.org/abs/2505.22937)  



*  Figure 1: EM and F1 grouped bar chart.


In [ ]:
plot_df = ablation_summary_df.melt(
    id_vars="Config",
    value_vars=["EM (%)", "F1 (%)"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=plot_df, x="Config", y="Score", hue="Metric")

# Add value labels on top of each bar
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=3, fontsize=9)

plt.title("EM and F1 Score Results Per Ablation Configuration", fontsize=13, fontweight="bold")
plt.ylabel("Score (%)")
plt.xlabel("Freeze Configuration")
plt.ylim(0, 100)
plt.xticks(rotation=15)
plt.legend(title="Metric")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("figure1_em_f1.png", dpi=150)
plt.show()
print("\nSaved into image 'figure1_em_f1.png'")

*  Figure 2: Training time and latency grouped bar chart.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Training time
ax1 = sns.barplot(
    data=ablation_summary_df,
    x="Config",
    y="Total Train Time (hrs)",
    ax=axes[0],
    color="#4C72B0"
)
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%.2f", padding=3, fontsize=9)

axes[0].set_title("Total Training Time by Config", fontweight="bold")
axes[0].set_ylabel("Training Time (hrs)")
axes[0].set_xlabel("Freeze Configuration")
axes[0].tick_params(axis="x", rotation=15)
axes[0].grid(axis="y", alpha=0.3)

# Inference latency
ax2 = sns.barplot(
    data=ablation_summary_df,
    x="Config",
    y="Inference Latency (ms)",
    ax=axes[1],
    color="#DD8452"
)
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%.2f", padding=3, fontsize=9)

axes[1].set_title("Inference Latency by Config", fontweight="bold")
axes[1].set_ylabel("Latency (ms)")
axes[1].set_xlabel("Freeze Configuration")
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("Training Cost and Inference Cost by Freeze Configuration",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("figure2_time_latency.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nSaved into image 'figure2_time_latency.png'")

*  Figure 3: Trainable parameters bar chart
This figure shows how many parameters remain trainable under each freezing configuration.  


In [ ]:
plt.figure(figsize=(10, 5))
ax = sns.barplot(
    data=ablation_summary_df,
    x="Config",
    y="Trainable Params (M)",
    color="#4C72B0"
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3, fontsize=9)

plt.title("Trainable Parameters by Freeze Configuration", fontsize=13, fontweight="bold")
plt.ylabel("Trainable Parameters (M)")
plt.xlabel("Freeze Configuration")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("figure3_trainable_params.png", dpi=150)
plt.show()
print("\nSaved into image 'figure3_trainable_params.png'")


*  Figure 4: F1 vs latency scatter plot with labels

In [ ]:
plt.figure(figsize=(8, 5))

for _, row in ablation_summary_df.iterrows():
    plt.scatter(
        row["Inference Latency (ms)"],
        row["F1 (%)"],
        s=120,
        zorder=5
    )
    plt.annotate(
        row["Config"],
        xy=(row["Inference Latency (ms)"], row["F1 (%)"]),
        xytext=(6, 4),
        textcoords="offset points",
        fontsize=9
    )

plt.title("F1 vs Inference Latency", fontsize=13, fontweight="bold")
plt.xlabel("Inference Latency (ms)")
plt.ylabel("F1 Score (%)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figure4_f1_vs_latency.png", dpi=150)
plt.show()
print("\nSaved into image 'figure4_f1_vs_latency.png'")

*  Figure 5: Radar chart<br>
It will include normalized values for EM, F1, training time, latency, and trainable parameters so that different scales can be compared on the same plot.  


In [ ]:
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

metrics = ["EM (%)", "F1 (%)", "Trainable Params (M)", "Total Train Time (hrs)", "Inference Latency (ms)"]
metric_labels = ["EM", "F1", "Trainable\nParams", "Train\nTime", "Latency"]

# Normalize each metric to [0, 1]
# For params, time, latency: lower is better so we invert
normalized = ablation_summary_df[metrics].copy()

for col in metrics:
    col_min = normalized[col].min()
    col_max = normalized[col].max()
    if col_max == col_min:
        normalized[col] = 1.0
    else:
        normalized[col] = (normalized[col] - col_min) / (col_max - col_min)

# Invert cost metrics so that higher = better on all axes
for col in ["Trainable Params (M)", "Total Train Time (hrs)", "Inference Latency (ms)"]:
    normalized[col] = 1 - normalized[col]

# Radar setup
num_metrics = len(metrics)
angles = np.linspace(0, 2 * np.pi, num_metrics, endpoint=False).tolist()
angles += angles[:1]   # close the polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

for i, row in ablation_summary_df.iterrows():
    values = normalized.iloc[i].tolist()
    values += values[:1]   # close the polygon
    ax.plot(angles, values, linewidth=1.8, label=row["Config"], color=colors[i])
    ax.fill(angles, values, alpha=0.1, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_labels, fontsize=10)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], fontsize=7)
ax.set_title("Normalized Multi-Metric Comparison\n(higher = better on all axes)",
             fontsize=12, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=9)

plt.tight_layout()
plt.savefig("figure5_radar.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nSaved into image 'figure5_radar.png'")

In [ ]:
def get_random_subset(dataset, fraction, seed):
    """
    Random sampling of dataset.
    """
    n_total = len(dataset)
    n_select = int(n_total * fraction)
    
    rng = np.random.default_rng(seed)
    indices = rng.choice(n_total, size=n_select, replace=False).tolist()
    return dataset.select(indices)


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np
import time
import pandas as pd

# Optimization HyperParam
LEARNING_RATE = 1e-5
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
NUM_EPOCHS = 2
MAX_SEQ_LENGTH = 384
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
SEED = 42
DATA_SIZES = [0.25, 0.50, 0.70]
N_RUNS = 3
SEEDS = [42, 123, 7]
val_examples = get_validation_examples(val_articles, limit=EVAL_LIMIT)

latency_sample = val_examples[0]
datasize_results = {}

for size in DATA_SIZES:
    run_em, run_f1, run_time, run_latency = [], [], [], []

    for run_idx, run_seed in enumerate(SEEDS):
        config_name = f"random_{int(size*100)}pct_run{run_idx+1}"
        print(f"\n{'='*60}")
        print(f"  Sampling=random | Size={int(size*100)}% | Run={run_idx+1}/3")
        print(f"{'='*60}")

        # Subset selection - using random sampling
        subset = get_random_subset(train_dataset, size, run_seed)
        print(f"  Subset size: {len(subset)} examples")

        # Load fresh model from HuggingFace
        model = DistilBertForQuestionAnswering.from_pretrained(
            "distilbert-base-uncased"
        ).to(DEVICE)

        args = TrainingArguments(
            output_dir = f"./datasize_{config_name}",
            eval_strategy = "epoch",
            save_strategy = "epoch",
            learning_rate = LEARNING_RATE,
            per_device_train_batch_size = BATCH_SIZE,
            per_device_eval_batch_size = BATCH_SIZE,
            gradient_accumulation_steps = GRAD_ACCUM_STEPS,
            num_train_epochs = NUM_EPOCHS,
            weight_decay = WEIGHT_DECAY,
            warmup_ratio = WARMUP_RATIO,
            load_best_model_at_end = True,
            metric_for_best_model = "eval_loss",
            seed = run_seed,
            logging_steps = 50,
            report_to = "none",
            fp16 = True,
        )

        trainer = Trainer(
            model = model,
            args = args,
            train_dataset = subset,
            eval_dataset = val_dataset,
            data_collator = default_data_collator,
        )

        # Training
        t_start = time.time()
        trainer.train()
        train_time_hrs = (time.time() - t_start) / 3600
        per_epoch_time_sec = (time.time() - t_start) / NUM_EPOCHS

        print(f"  Total training time : {train_time_hrs:.2f} hrs")
        print(f"  Avg time per epoch  : {per_epoch_time_sec:.2f} sec")

        # Latency 
        model.eval()
        inputs = tokenizer(
            latency_sample["question"],
            latency_sample["context"],
            max_length = 384,
            truncation = "only_second",
            return_tensors = "pt"
        ).to(DEVICE)

        with torch.no_grad():
            _ = model(**inputs)
            t_lat = time.time()
            _ = model(**inputs)
            latency_ms = (time.time() - t_lat) * 1000

        print(f"  Inference latency   : {latency_ms:.2f} ms")

        # Evaluation
        qa_pipeline = pipeline(
            "question-answering",
            model = model,
            tokenizer = tokenizer,
            device = 0 if torch.cuda.is_available() else -1
        )

        em_scores, f1_scores = [], []
        for ex in val_examples:
            output = qa_pipeline(
                question = ex["question"],
                context = ex["context"],
                max_answer_len = 40
            )
            em, f1 = score_prediction(ex["gold_answers"], output["answer"])
            em_scores.append(em)
            f1_scores.append(f1)

        run_em_val = np.mean(em_scores) * 100
        run_f1_val = np.mean(f1_scores) * 100

        print(f"  EM : {run_em_val:.2f}%")
        print(f"  F1 : {run_f1_val:.2f}%")

        run_em.append(run_em_val)
        run_f1.append(run_f1_val)
        run_time.append(train_time_hrs)
        run_latency.append(latency_ms)

        # Save per-run result 
        datasize_results[config_name] = {
            "sampling": "random",
            "data_size_pct": int(size * 100),
            "run": run_idx + 1,
            "subset_size": len(subset),
            "em": run_em_val,
            "f1": run_f1_val,
            "train_time_hrs": train_time_hrs,
            "avg_epoch_time_sec": per_epoch_time_sec,
            "latency_ms": latency_ms,
        }

        del model, trainer, qa_pipeline
        torch.cuda.empty_cache()

    # Aggregate across 3 runs
    agg_key = f"random_{int(size*100)}pct_MEAN"
    datasize_results[agg_key] = {
        "sampling": "random",
        "data_size_pct": int(size * 100),
        "run": "mean±std",
        "subset_size": "-",
        "em": f"{np.mean(run_em):.2f} ± {np.std(run_em):.2f}",
        "f1": f"{np.mean(run_f1):.2f} ± {np.std(run_f1):.2f}",
        "train_time_hrs": f"{np.mean(run_time):.2f} ± {np.std(run_time):.2f}",
        "latency_ms": f"{np.mean(run_latency):.2f} ± {np.std(run_latency):.2f}",
    }

    print(f"\n  => Aggregated (random {int(size*100)}%) ──")
    print(f"  EM : {datasize_results[agg_key]['em']}")
    print(f"  F1 : {datasize_results[agg_key]['f1']}")

datasize_df = pd.DataFrame(datasize_results).T.reset_index(drop=True)
datasize_df.to_csv("datasize_ablation_results.csv", index=False)

print("\n✅ Data size ablation complete")
print(datasize_df.to_string(index=False))

In [ ]:
# Figure 1: EM and F1 across different data sizes
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Extract mean values for each data size from datasize_results
summary_data = []
for size in [25, 50, 70]:
    agg_key = f"random_{size}pct_MEAN"
    if agg_key in datasize_results:
        agg_data = datasize_results[agg_key]
        summary_data.append({
            "Data Size": f"{size}%",
            "EM (%)": float(agg_data['em'].split(' ± ')[0]),
            "F1 (%)": float(agg_data['f1'].split(' ± ')[0])
        })

summary_df = pd.DataFrame(summary_data)

# Melt for seaborn
plot_df = summary_df.melt(
    id_vars="Data Size",
    value_vars=["EM (%)", "F1 (%)"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=plot_df, x="Data Size", y="Score", hue="Metric")

# Add value labels
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=3, fontsize=10)

plt.title("EM and F1 Scores Across Different Training Data Sizes", 
          fontsize=14, fontweight="bold")
plt.ylabel("Score (%)", fontsize=12)
plt.xlabel("Training Data Size", fontsize=12)
plt.ylim(0, 100)
plt.legend(title="Metric", title_fontsize=11, fontsize=10)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("figure1_em_f1_datasize.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✅ Figure 1 saved as 'figure1_em_f1_datasize.png'")
print("\nSummary Table:")
print(summary_df.to_string(index=False))

In [ ]:
# Figure 2: Training time and latency across data sizes
time_latency_data = []
for size in [25, 50, 70]:
    agg_key = f"random_{size}pct_MEAN"
    if agg_key in datasize_results:
        agg_data = datasize_results[agg_key]
        time_latency_data.append({
            "Data Size": f"{size}%",
            "Total Train Time (hrs)": float(agg_data['train_time_hrs'].split(' ± ')[0]),
            "Inference Latency (ms)": float(agg_data['latency_ms'].split(' ± ')[0])
        })

time_latency_df = pd.DataFrame(time_latency_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training time subplot
ax1 = sns.barplot(
    data=time_latency_df,
    x="Data Size",
    y="Total Train Time (hrs)",
    ax=axes[0],
    color="#4C72B0"
)
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%.2f", padding=3, fontsize=10)

axes[0].set_title("Total Training Time by Data Size", fontweight="bold", fontsize=12)
axes[0].set_ylabel("Training Time (hrs)", fontsize=11)
axes[0].set_xlabel("Training Data Size", fontsize=11)
axes[0].grid(axis="y", alpha=0.3)

# Inference latency subplot
ax2 = sns.barplot(
    data=time_latency_df,
    x="Data Size",
    y="Inference Latency (ms)",
    ax=axes[1],
    color="#DD8452"
)
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%.2f", padding=3, fontsize=10)

axes[1].set_title("Inference Latency by Data Size", fontweight="bold", fontsize=12)
axes[1].set_ylabel("Latency (ms)", fontsize=11)
axes[1].set_xlabel("Training Data Size", fontsize=11)
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("Training Cost and Inference Cost by Training Data Size",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("figure2_time_latency_datasize.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✅ Figure 2 saved as 'figure2_time_latency_datasize.png'")
print("\nTraining Time & Latency Table:")
print(time_latency_df.to_string(index=False))

In [ ]:
# Figure 3: Number of training examples per data size
# Extract subset sizes from first run of each size
subset_data = []
for size in [25, 50, 70]:
    config_name = f"random_{size}pct_run1"
    if config_name in datasize_results:
        subset_data.append({
            "Data Size": f"{size}%",
            "Training Examples": datasize_results[config_name]['subset_size']
        })

subset_df = pd.DataFrame(subset_data)

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=subset_df,
    x="Data Size",
    y="Training Examples",
    color="#55A868"
)

for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3, fontsize=11)

plt.title("Number of Training Examples by Data Size", fontsize=14, fontweight="bold")
plt.ylabel("Number of Training Examples", fontsize=12)
plt.xlabel("Data Size (% of Full Dataset)", fontsize=12)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("figure3_dataset_size.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✅ Figure 3 saved as 'figure3_dataset_size.png'")
print("\nDataset Size Table:")
print(subset_df.to_string(index=False))

In [ ]:
# Figure 4: F1 vs Training Time scatter plot (trade-off analysis)
scatter_data = []
for size in [25, 50, 70]:
    agg_key = f"random_{size}pct_MEAN"
    if agg_key in datasize_results:
        agg_data = datasize_results[agg_key]
        scatter_data.append({
            "Data Size": f"{size}%",
            "F1 Score": float(agg_data['f1'].split(' ± ')[0]),
            "Training Time (hrs)": float(agg_data['train_time_hrs'].split(' ± ')[0])
        })

scatter_df = pd.DataFrame(scatter_data)

plt.figure(figsize=(10, 7))

# Create scatter plot
for _, row in scatter_df.iterrows():
    plt.scatter(
        row["Training Time (hrs)"],
        row["F1 Score"],
        s=200,
        zorder=5,
        marker='o'
    )
    plt.annotate(
        row["Data Size"],
        xy=(row["Training Time (hrs)"], row["F1 Score"]),
        xytext=(8, 5),
        textcoords="offset points",
        fontsize=11,
        fontweight="bold"
    )

plt.title("F1 Score vs Training Time Trade-off\n(Performance vs Computational Cost)", 
          fontsize=14, fontweight="bold")
plt.xlabel("Training Time (hours)", fontsize=12)
plt.ylabel("F1 Score (%)", fontsize=12)
plt.grid(alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig("figure4_f1_vs_training_time.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✅ Figure 4 saved as 'figure4_f1_vs_training_time.png'")
print("\nTrade-off Analysis Table:")
print(scatter_df.to_string(index=False))

In [ ]:
# Figure 5: Radar chart comparing all metrics across data sizes
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

# Metrics to compare
metrics = ["em", "f1", "train_time_hrs", "latency_ms"]
metric_labels = ["EM", "F1", "Train\nTime", "Latency"]

# Collect data for each size
radar_data = {}
for size in [25, 50, 70]:
    agg_key = f"random_{size}pct_MEAN"
    if agg_key in datasize_results:
        agg_data = datasize_results[agg_key]
        radar_data[f"{size}%"] = {
            "em": float(agg_data['em'].split(' ± ')[0]),
            "f1": float(agg_data['f1'].split(' ± ')[0]),
            "train_time_hrs": float(agg_data['train_time_hrs'].split(' ± ')[0]),
            "latency_ms": float(agg_data['latency_ms'].split(' ± ')[0])
        }

# Normalize each metric to [0, 1]
normalized_data = {}
for size, metrics_dict in radar_data.items():
    normalized = {}
    for metric in metrics:
        values = [radar_data[s][metric] for s in radar_data.keys()]
        min_val = min(values)
        max_val = max(values)
        if max_val == min_val:
            normalized[metric] = 1.0
        else:
            normalized[metric] = (metrics_dict[metric] - min_val) / (max_val - min_val)
    
    # Invert cost metrics (lower is better)
    for metric in ["train_time_hrs", "latency_ms"]:
        normalized[metric] = 1 - normalized[metric]
    
    normalized_data[size] = normalized

# Radar setup
num_metrics = len(metrics)
angles = np.linspace(0, 2 * np.pi, num_metrics, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

colors = ["#4C72B0", "#DD8452", "#55A868"]
markers = ['o', 's', '^']

for i, (size, norm_metrics) in enumerate(normalized_data.items()):
    values = [norm_metrics[metric] for metric in metrics]
    values += values[:1]
    ax.plot(angles, values, linewidth=2, label=f"{size} Data", color=colors[i], marker=markers[i], markersize=8)
    ax.fill(angles, values, alpha=0.1, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], fontsize=9)
ax.set_title("Normalized Multi-Metric Comparison Across Data Sizes\n(higher = better on all axes)",
             fontsize=13, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=10)

plt.tight_layout()
plt.savefig("figure5_radar_datasize.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✅ Figure 5 saved as 'figure5_radar_datasize.png'")

### 🔷 <font color="#9CE2FF">Sherouk's Work Ends Here</font>

In [ ]:
import shutil
import os

os.makedirs("/kaggle/working/transfer", exist_ok=True)

# Copy trained models
for folder in ["distilbert-squad-final", "tinybert-squad-final"]:
    if os.path.exists(f"./{folder}"):
        shutil.copytree(f"./{folder}", f"/kaggle/working/transfer/{folder}")
        print(f"Copied {folder} ✅")

# Copy all CSVs, PNGs, JSONs
for f in os.listdir("."):
    if f.endswith((".csv", ".json", ".png", ".py")):
        shutil.copy(f, "/kaggle/working/transfer/")
        print(f"Copied {f} ✅")

# Zip everything
shutil.make_archive("/kaggle/working/transfer_package", "zip", "/kaggle/working/transfer")
print("\nDone ✅ — download transfer_package.zip from Output panel")